# Results for model: deepseek-v3.2

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Feature engineering: Calculate rolling top 15% of feature_00 relative to responder_6
df = df.with_columns([
    pl.col('feature_00').rolling_quantile(
        quantile=0.85,
        window_size=pl.col('date_id').count(),
        min_periods=1,
        by='date_id'
    ).over('date_id').alias('feature_00_85th')
])

df = df.with_columns([
    (pl.col('feature_00') > pl.col('feature_00_85th')).cast(pl.Int8).alias('is_top_15')
])

# Prepare features and target
feature_cols = [f'feature_{i:02d}' for i in range(77)] + ['is_top_15']
X = df.select(feature_cols).fill_null(0).to_numpy()
y = df.select('responder_6').fill_null(0).to_numpy().ravel()

# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
model.fit(X, y)